# Praktikum 07 – Fine-tuning mit PEFT und QLoRA
**Applied Generative AI – NLP | Sommersemester 2026**

**Lernziele:** RAG vs. Fine-tuning Entscheidung verstehen · LoRA-Parameter (rank, alpha, target_modules) konfigurieren · QLoRA mit 4-bit NF4-Quantisierung anwenden · Modell auf spezifisches Verhalten finetunen · Vorher/Nachher-Vergleich durchführen.

**Zielprodukt nach 90 Minuten:**
1. Eine Entscheidungsmatrix, wann RAG oder Fine-tuning sinnvoller ist.
2. Ein vortrainiertes Modell mit LoRA-Adaptern aufgabenspezifisch optimieren.
3. Einen quantifizierbaren Vorher/Nachher-Vergleich der Modellantworten.

---
```bash
uv pip install --system transformers torch peft bitsandbytes accelerate sentencepiece protobuf flash-attn
```

**Hinweis:** Dieses Notebook verwendet Hugging Face transformers + PEFT (nicht Ollama), da PEFT-Fine-tuning die direkte Modellkontrolle erfordert.

In [ ]:
import importlib
import os
import shutil
import subprocess
import sys
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules

# Modell-Konfiguration - kleines Modell für schnelles Training
MODEL_NAME = os.getenv("HF_MODEL", "Qwen/Qwen2.5-0.5B-Instruct").strip()
MODEL_REVISION = os.getenv("HF_REVISION", "main").strip()

REQUIRED = {
    "transformers": ("transformers", "4.47.0"),
    "torch": ("torch", "2.5.1"),
    "peft": ("peft", "0.14.0"),
    "bitsandbytes": ("bitsandbytes", "0.44.1"),
    "accelerate": ("accelerate", "1.2.1"),
    "sentencepiece": ("sentencepiece", "0.2.0"),
    "protobuf": ("protobuf", "5.29.1"),
}


def run_install(specs):
    if shutil.which("uv") is None:
        raise RuntimeError("uv is not installed. Install uv and rerun the setup cell.")

    command = ["uv", "pip", "install", "--python", sys.executable, *specs]
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix) or bool(os.environ.get("VIRTUAL_ENV"))
    if not in_venv:
        command.append("--system")

    print("$", " ".join(command))
    subprocess.check_call(command)


missing_specs = [
    f"{install_name}=={version}"
    for install_name, (import_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name, _ in REQUIRED if find_spec(import_name) is None]
if missing_after:
    raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import transformers
import torch
import peft
import bitsandbytes
import accelerate

print("Runtime:", "Google Colab" if IN_COLAB else "Lokal/Jupyter")
print("Model:", MODEL_NAME)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## Teil 1 – RAG vs. Fine-tuning Entscheidungsmatrix (15 min)

Wann sollte man RAG verwenden und wann ist Fine-tuning die bessere Wahl? Diese Entscheidung beeinflusst den Projekterfolg massgeblich.

**Pflichtschritte:**
- Analysiere die Vor- und Nachteile beider Ansätze.
- Erstelle eine tabellarische Entscheidungshilfe mit konkreten Anwendungsfällen.
- Begründe anhand von Beispielen, wann welcher Ansatz sinnvoll ist.

**Soll-Ergebnis:** Eine Übersichtstabelle und eigene Entscheidungskriterien.

In [ ]:
# Daten für die Entscheidungsmatrix
import pandas as pd

decision_matrix = pd.DataFrame({
    "Kriterium": [
        "Wissensaktualität",
        "Domänenspezifisches Vokabular",
        "Antwortstil / Persona",
        "Rechenaufwand",
        "Datenschutz / Vertraulichkeit",
        "Latenz-Anforderungen",
        "Komplexität der Implementierung",
        "Kosten (Training vs. Retrieval)"
    ],
    "RAG (empfohlen)": [
        "✓ Hoch - neues Wissen schnell integrierbar",
        "✓ Gut - externe Dokumente mit Fachbegriffen",
        "✗ Eingeschränkt - nur Prompts",
        "✓ Niedrig - nur Inference + Embedding",
        "✓ Daten können lokal bleiben",
        "⚠ Mittel - Retrieval adds overhead",
        "✓ Einfach - schnell aufsetzbar",
        "✓ Niedrig bis mittel"
    ],
    "Fine-tuning (empfohlen)": [
        "✗ Niedrig - Neustraining erforderlich",
        "✓ Hervorragend - lernt Domänen-spezifisch",
        "✓ Hervorragend - Stil und Persona einprogrammiert",
        "✗ Hoch - Training erforderlich",
        "✗ Problematisch - Trainingsdaten müssen zentralisiert werden",
        "✓ Niedrig - kein Retrieval nötig",
        "✗ Komplexer - erfordert ML-Kenntnisse",
        "✗ Mittel bis hoch"
    ],
    "Typischer Anwendungsfall": [
        "Wissensdatenbank mit häufigen Updates",
        "Medizinische Texte, Rechtsterminologie",
        "Kundenservice mit spezifischem Ton",
        "Batch-Verarbeitung, hohe Throughput",
        "Interne Dokumente, kein External Hosting",
        "Echtzeit-Antworten, Chatbot",
        "Proof-of-Concept → Produktion",
        "Kleines Team, begrenztes Budget"
    ]
})

print("=" * 80)
print("ENTSCHEIDUNGSMATRIX: RAG vs. Fine-tuning")
print("=" * 80)
print(decision_matrix.to_string(index=False))
print("\n" + "=" * 80)
print("FAZIT: RAG eignet sich für dynamisches Wissen, Fine-tuning für stabile Verhaltensweisen.")
print("Oft ist eine Kombination (RAG + Fine-tuning) optimal.")
print("=" * 80)

## Teil 2 – LoRA-Mathematik und Konfiguration (20 min)

LoRA (Low-Rank Adaptation) ist eine Parameter-effiziente Fine-tuning-Methode, die nur kleine Matrizen hinzufügt, anstatt das gesamte Modell zu trainieren.

### Die LoRA-Mathematik

Für eine Attention-Gewichtsmatrix $W \in \mathbb{R}^{d \times k}$ wird folgende Zerlegung trainiert:

$$h = W \cdot x + \frac{\alpha}{r} \cdot B \cdot A \cdot x$$

Dabei ist:
- $A \in \mathbb{R}^{r \times d}$ und $B \in \mathbb{R}^{k \times r}$ sind die neuen, trainierbaren Matrizen
- $r$ ist der **Rank** (Rang) - bestimmt die Kapazität der Anpassung
- $\alpha$ ist ein Skalierungsfaktor, der die Stärke der Anpassung kontrolliert
- Typische Werte: $r \in \{8, 16, 32, 64\}$, $\alpha \approx 2 \times r$

**Pflichtschritte:**
- Verstehe die LoRA-Formel und ihre Parameter.
- Konfiguriere target_modules (welche Schichten werden adaptiert?).
- Experimentiere mit verschiedenen r-Werten (theoretisch).

**Soll-Ergebnis:** Eine konfigurierte LoRA-Config mit dokumentierten Parametern.

In [ ]:
from peft import LoraConfig, TaskType

# LoRA-Konfiguration erklärt
lora_config_explanation = """
LoRA-KONFIGURATION - PARAMETER-ERKLÄRUNG
=========================================

1. rank (r):
   - Dimension der Low-Rank-Matrizen
   - Klein = schneller, weniger Speicher, aber weniger Kapazität
   - Gross = mehr Kapazität, aber mehr Speicher und Trainingzeit
   - Empfehlung: 8-32 für kleine Modelle, 64-128 für grosse

2. lora_alpha:
   - Skalierungsfaktor für die LoRA-Anpassung
   - Regel: lora_alpha ≈ 2 * rank (oft funktioniert gut)
   - Höher = stärkere Anpassung, niedriger = subtilere Änderungen

3. target_modules:
   - Welche Attention-Matrizen werden adaptiert?
   - q_proj, v_proj = Query und Value (am wichtigsten)
   - k_proj, o_proj = Key und Output (optional)
   -Für Qwen2: ['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

4. lora_dropout:
   - Dropout-Rate für LoRA-Gewichte (Regularisierung)
   - Meist 0.0-0.1, 0.05 ist guter Standard

5. bias:
   - 'none': keine Bias-Trainierung
   - 'all': trainiert auch Bias
   - 'lora_only': nur LoRA-Bias
   - Empfehlung: 'none' für die meisten Fälle

6. task_type:
   - TASK_TYPE_CAUSAL_LM für Causal Language Models (Autoregressive)
   - TASK_TYPE_SEQ_CLS für Sequence Classification
"""

print(lora_config_explanation)

# Praxis: LoRA-Config für unser Qwen-Modell erstellen
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                    # Rank: 16 - guter Kompromiss
    lora_alpha=32,           # Alpha: 2 * r
    lora_dropout=0.05,      # Leichter Dropout
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    bias="none",
    inference_mode=False,    # Training-Modus
)

print("\n" + "=" * 60)
print("KONFIGURIERTE LORA-CONFIG:")
print("=" * 60)
print(f"Task Type: {lora_config.task_type}")
print(f"Rank (r): {lora_config.r}")
print(f"Alpha: {lora_config.lora_alpha}")
print(f"Dropout: {lora_config.lora_dropout}")
print(f"Target Modules: {lora_config.target_modules}")
print(f"Bias: {lora_config.bias}")
print("=" * 60)

# Berechnung der Parameter-Einsparung (theoretisch)
original_params = 500_000_000  # Ca. 0.5B für Qwen2.5-0.5B
lora_params = 7 * 2 * 16 * 2048  # 7 Module * 2 Matrizen (A,B) * r * hidden_size
print(f"\nGeschätzte LoRA-Parameter: ~{lora_params:,}")
print(f"Original-Modell: ~{original_params:,}")
print(f"Einsparung: {100 * (1 - lora_params/original_params):.2f}%")

## Teil 3 – QLoRA mit 4-bit NF4-Quantisierung (15 min)

QLoRA kombiniert LoRA mit Quantisierung, um noch speichereffizienter zu trainieren. Die NF4-Quantisierung (Normal Float 4) ist optimiert für die Kompression von neuronalen Netzgewichten.

### Wie funktioniert NF4-Quantisierung?

4-bit NF4 (Normal Float 4) nutzt eine quantisierte Darstellung mit 16 möglichen Werten, die optimal an die Normalverteilung von Gewichten angepasst ist:

1. **Normalisierung**: Gewichte werden auf [-1, 1] normiert
2. **Quantisierung**: Kontinuierliche Werte → 4-bit diskrete Stufen
3. **Dequantisierung**: Zur Inference werden die Werte rekonstruiert

**Vorteile:**
- ~75% Speicherersparnis (32-bit → 4-bit)
- Originale Gewichte bleiben erhalten (nur Kopie wird quantisiert)
- LoRA-Adapter werden in voller Präzision trainiert

**Pflichtschritte:**
- Konfiguriere BitsAndBytesConfig für NF4-Quantisierung.
- Lade das Modell mit 4-bit-Quantisierung.
- Überprüfe die erfolgreiche Quantisierung.

**Soll-Ergebnis:** Ein quantisiertes Modell, bereit für Training.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# QLoRA-Konfiguration mit NF4-Quantisierung
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # 4-bit Quantisierung aktivieren
    bnb_4bit_quant_type="nf4",            # NF4 (Normal Float 4)
    bnb_4bit_compute_dtype=torch.float16, # Rechenpräzision
    bnb_4bit_use_double_quant=True,       # Double-Quantisierung für mehr Kompression
)

print("=" * 60)
print("QLORA-KONFIGURATION - 4-bit NF4 Quantisierung")
print("=" * 60)
print(f"load_in_4bit: {quantization_config.load_in_4bit}")
print(f"quant_type: {quantization_config.bnb_4bit_quant_type}")
print(f"compute_dtype: {quantization_config.bnb_4bit_compute_dtype}")
print(f"double_quant: {quantization_config.bnb_4bit_use_double_quant}")
print("=" * 60)

print("\nLade Modell mit QLoRA-Konfiguration...")
print("(Dies kann je nach Internetverbindung einige Minuten dauern)")

# Modell und Tokenizer laden
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    padding_side="right",
)

# Sicherstellen, dass pad_token gesetzt ist
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

print("\n✓ Modell erfolgreich geladen mit 4-bit NF4 Quantisierung")
print(f"Modell-Typ: {type(model).__name__}")
print(f"Modell-Grösse (geschätzt): ~{model.get_memory_footprint() / 1e9:.2f} GB")

# Prüfen, ob Modell tatsächlich quantisiert ist
# (Optionale Information - nicht kritisch für Notebook-Funktionalität)
if find_spec("bitsandbytes.nn") is not None:
    from bitsandbytes.nn import Linear4bit
    quantized_layers = sum(1 for m in model.modules() if isinstance(m, Linear4bit))
    print(f"Quantisierte Schichten: {quantized_layers}")
else:
    print("Hinweis: bitsandbytes 4-bit Schichten können nicht direkt gezählt werden.")

print("\nModell bereit für PEFT-Training!")

## Teil 4 – Training durchführen (25 min)

Jetzt wenden wir die LoRA-Adapter auf das quantisierte Modell an und führen ein kurzes Training durch.

### Trainingsdaten

Wir erstellen einen kleinen Datensatz, der das Modell darauf trainiert, wie ein hilfreicher KI-Assistent zu antworten (ähnlich dem Instruct-Tuning).

**Pflichtschritte:**
- Bereite Trainingsdaten vor (kleiner, aber aussagekräftiger Datensatz).
- Wende PEFT-Model auf das quantisierte Modell an.
- Führe einen kurzen Trainingsepochus durch.
- Speichere das trainierte Modell.

**Soll-Ergebnis:** Ein feinabgestimmtes Modell mit LoRA-Adaptern.

In [ ]:
from peft import get_peft_model
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset

# Kleiner Trainingsdatensatz - instruktionsartige Dialoge
training_data = [
    {
        "instruction": "Du bist ein hilfreicher Assistent. Beantworte die Frage kurz und präzise.",
        "input": "Was ist Python?",
        "output": "Python ist eine interpretierte, höhere Programmiersprache mit einfach lesbarer Syntax. Sie wird häufig für Webentwicklung, Datenanalyse und maschinelles Lernen verwendet."
    },
    {
        "instruction": "Du bist ein hilfreicher Assistent. Beantworte die Frage kurz und präzise.",
        "input": "Was ist maschinelles Lernen?",
        "output": "Maschinelles Lernen ist ein Teilbereich der KI, bei dem Algorithmen aus Daten lernen, um Vorhersagen oder Entscheidungen zu treffen, ohne explizit programmiert zu werden."
    },
    {
        "instruction": "Du bist ein hilfreicher Assistent. Beantworte die Frage kurz und präzise.",
        "input": "Was ist ein Transformer?",
        "output": "Ein Transformer ist eine neuronale Netzwerkarchitektur, die Self-Attention verwendet, um Kontext in Sequenzen zu erfassen. Er bildet die Grundlage moderner grosser Sprachmodelle."
    },
    {
        "instruction": "Du bist ein hilfreicher Assistent. Beantworte die Frage kurz und präzise.",
        "input": "Was ist RAG?",
        "output": "RAG (Retrieval Augmented Generation) kombiniert einen Retrieval-Mechanismus mit einem generativen Modell, um Antworten mit aktuellen oder spezifischen Informationen zu verbessern."
    },
    {
        "instruction": "Du bist ein hilfreicher Assistent. Beantworte die Frage kurz und präzise.",
        "input": "Was ist LoRA?",
        "output": "LoRA (Low-Rank Adaptation) ist eine effiziente Methode zum Fine-tuning, die trainierbare Low-Rank-Matrizen zu einem vortrainierten Modell hinzufügt, statt das gesamte Modell neu zu trainieren."
    },
    {
        "instruction": "Du bist ein hilfreicher Assistent. Beantworte die Frage kurz und präzise.",
        "input": "Was ist Fine-tuning?",
        "output": "Fine-tuning ist die Methode, ein bereits vortrainiertes Modell auf spezifische Aufgaben oder Domänen anzupassen, indem man es mit zusätzlichen Daten weiter trainiert."
    },
    {
        "instruction": "Du bist ein hilfreicher Assistent. Beantworte die Frage kurz und präzise.",
        "input": "Was sind Embeddings?",
        "output": "Embeddings sind dichte Vektordarstellungen von Wörtern oder Texten, die semantische Bedeutungen in numerischer Form erfassen und ähnliche Konzepte im Vektorraum nahe beieinander platzieren."
    },
    {
        "instruction": "Du bist ein hilfreicher Assistent. Beantworte die Frage kurz und präzise.",
        "input": "Was ist ein Vektordatenbank?",
        "output": "Eine Vektordatenbank speichert Embeddings und ermöglicht schnelle Ähnlichkeitssuchen. Sie wird für Retrieval-Augmented Generation und semantische Suche verwendet."
    },
]

print(f"Trainingsdatensatz: {len(training_data)} Beispiele")

# Daten formatieren für das Modell
def format_example(example):
    # Format: Instruction + Input -> Output
    text = f"### Instruction\n{example['instruction']}\n\n### Input\n{example['input']}\n\n### Response\n{example['output']}"
    return {"text": text}

formatted_data = [format_example(d) for d in training_data]

# Dataset erstellen
dataset = Dataset.from_list(formatted_data)

# Tokenisierung
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print(f"Tokenisierte Beispiele: {len(tokenized_dataset)}")
print(f"Tokenizer-Max-Länge: {tokenizer.model_max_length}")
print(f"Beispiel-Input-IDs (erste 50): {tokenized_dataset[0]['input_ids'][:50]}...")

In [ ]:
# PEFT-Modell erstellen (LoRA-Adapter hinzufügen)
peft_model = get_peft_model(model, lora_config)

print("=" * 60)
print("PEFT-MODELL MIT LORA-ADAPTERN")
print("=" * 60)

# Zählen der trainierbaren Parameter
peft_model.print_trainable_parameters()

print("\n" + "=" * 60)
print("DETAILS DER LORA-ADAPTER")
print("=" * 60)

# LoRA-Adapter anzeigen
lora_layers = []
for name, module in peft_model.named_modules():
    if "lora" in name.lower():
        lora_layers.append(name)

print(f"Anzahl LoRA-Module: {len(lora_layers)}")
print("Beispiel-LoRA-Modulnamen:")
for name in lora_layers[:5]:
    print(f"  - {name}")
print(f"  ... und {len(lora_layers) - 5} weitere")

print("\n✓ PEFT-Modell mit LoRA-Adaptern bereit für Training")

In [ ]:
# Training-Konfiguration
training_args = TrainingArguments(
    output_dir="./lora_output",
    num_train_epochs=3,              # 3 Epochen - kurz für Demo
    per_device_train_batch_size=2,  # Kleiner Batch für Speicher
    gradient_accumulation_steps=4,  # Effektiver Batch = 8
    learning_rate=3e-4,             # Typischer LR für LoRA
    logging_steps=1,                # Logging bei jedem Schritt
    save_steps=10,                  # Alle 10 Schritte speichern
    save_total_limit=1,             # Nur das beste Modell behalten
    warmup_steps=5,                 # Warmup für Stabilität
    report_to="none",               # Kein Logging-Service
    optim="adamw_torch",            # AdamW Optimizer
    fp16=False,                     # Im FP16-Modus (kommt von Quantisierung)
    remove_unused_columns=False,
)

# Data Collator für Language Modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,                      # Causal LM (nicht Masked LM)
)

print("=" * 60)
print("TRAINING-KONFIGURATION")
print("=" * 60)
print(f"Epochen: {training_args.num_train_epochs}")
print(f"Batch-Grösse (pro Gerät): {training_args.per_device_train_batch_size}")
print(f"Gradient Accumulation: {training_args.gradient_accumulation_steps}")
print(f"Effektive Batch-Grösse: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning Rate: {training_args.learning_rate}")
print(f"Warmup Steps: {training_args.warmup_steps}")
print("=" * 60)

# Training starten
from transformers import Trainer

print("\nStarte Training...")
print("(Dies dauert ca. 2-5 Minuten für diesen kleinen Datensatz)")

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Training durchführen
train_result = trainer.train()

print("\n" + "=" * 60)
print("TRAINING ABGESCHLOSSEN")
print("=" * 60)
print(f"Train Loss: {train_result.training_loss:.4f}")
print(f"Training Steps: {trainer.state.global_step}")
print("=" * 60)

In [ ]:
# LoRA-Adapter speichern
adapter_path = "./lora_adapters"
peft_model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("=" * 60)
print("LORA-ADAPTER GESPEICHERT")
print("=" * 60)
print(f"Pfad: {adapter_path}")
print("\nGespeicherte Dateien:")
import os
for f in os.listdir(adapter_path):
    filepath = os.path.join(adapter_path, f)
    size = os.path.getsize(filepath)
    print(f"  - {f}: {size:,} Bytes")

print("=" * 60)
print("✓ Adapter können später wieder geladen werden mit:")
print("  from peft import PeftModel")
print("  model = PeftModel.from_pretrained(base_model, './lora_adapters')")
print("=" * 60)

## Teil 5 – Vorher/Nachher-Vergleich (15 min)

Jetzt vergleichen wir die Antworten des Originalmodells mit dem feinabgestimmten Modell, um den Effekt des Trainings zu sehen.

**Pflichtschritte:**
- Definiere Testfragen, die das Gelernte abfragen.
- Generiere Antworten mit dem Originalmodell.
- Generiere Antworten mit dem feinabgestimmten Modell.
- Vergleiche und analysiere die Unterschiede.

**Soll-Ergebnis:** Dokumentierter Vorher/Nachher-Vergleich mit Bewertung.

In [ ]:
from peft import PeftModel

# Original-Modell (ohne LoRA) für Vergleich laden
print("Lade Original-Modell für Vergleich...")

# Wir verwenden das bereits geladene Modell als Basis
# Um fair zu vergleichen, erstellen wir eine Kopie ohne Adapter
from transformers import AutoModelForCausalLM

original_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

# Testfragen - diese wurden NICHT im Training verwendet
test_questions = [
    "Was ist ein grosses Sprachmodell?",
    "Erkläre den Begriff Attention-Mechanismus.",
    "Was versteht man unter Quantisierung?",
]

print("=" * 60)
print("VORHER/NACHHER VERGLEICH")
print("=" * 60)
print(f"Testfragen: {len(test_questions)}")
for i, q in enumerate(test_questions, 1):
    print(f"  {i}. {q}")
print("=" * 60)

In [ ]:
def generate_response(model, tokenizer, question, max_new_tokens=100):
    """Generiere eine Antwort auf eine Frage."""
    prompt = f"### Instruction\nDu bist ein hilfreicher Assistent. Beantworte die Frage kurz und präzise.\n\n### Input\n{question}\n\n### Response\n"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Nur den Response-Teil extrahieren
    if "### Response\n" in response:
        response = response.split("### Response\n")[-1]
    return response.strip()


print("Generiere Antworten mit ORIGINAL-Modell (ohne Fine-tuning)...")
print("-" * 60)

original_responses = {}
for question in test_questions:
    response = generate_response(original_model, tokenizer, question)
    original_responses[question] = response
    print(f"Frage: {question}")
    print(f"Antwort: {response[:200]}...")
    print("-" * 60)

In [ ]:
print("\nGeneriere Antworten mit FEINGETUNTEM Modell (mit LoRA)...")
print("-" * 60)

finetuned_responses = {}
for question in test_questions:
    response = generate_response(peft_model, tokenizer, question)
    finetuned_responses[question] = response
    print(f"Frage: {question}")
    print(f"Antwort: {response[:200]}...")
    print("-" * 60)

In [ ]:
# Vergleichstabelle
print("\n" + "=" * 80)
print("VERGLEICHSTABELLE: VORHER vs. NACHHER")
print("=" * 80)

for question in test_questions:
    print(f"\n📌 Frage: {question}")
    print("-" * 60)
    
    print("ORIGINAL (ohne Fine-tuning):")
    orig = original_responses[question][:300]
    print(f"  {orig}...")
    
    print("\nFEINGETUNT (mit LoRA):")
    tuned = finetuned_responses[question][:300]
    print(f"  {tuned}...")
    
    print("-" * 60)

print("\n" + "=" * 80)
print("ANALYSE")
print("=" * 80)
print("""
Beobachtungen beim Vergleich:

1. **Stilkonsistenz**: Das feinabgestimmte Modell sollte kürzere,
   präzisere Antworten geben (entsprechend unserem Trainingsformat).

2. **Struktur**: Das getunte Modell folgt dem vorgegebenen Format
   (Antwort direkt nach "Response:").

3. **Inhalt**: Beide Modelle haben dasselbe Wissen (gleiche Basisgewichte),
   aber das LoRA-Modell antwortet konsistenter im trainierten Stil.

4. **Hinweis**: Bei einem so kleinen Datensatz (8 Beispiele) und nur
   3 Epochen ist der Effekt subtil. Bei mehr Daten und Training
   werden die Unterschiede deutlicher sichtbar.
""")
print("=" * 80)

## Zusammenfassung und Ausblick

In diesem Praktikum haben wir folgende Themen behandelt:

### Gelernte Konzepte

1. **RAG vs. Fine-tuning**: Wann welcher Ansatz sinnvoll ist
2. **LoRA-Mathematik**: Low-Rank-Adaptation mit rang-r Matrizen
3. **QLoRA**: Kombination aus Quantisierung und LoRA für Effizienz
4. **PEFT**: Parameter-Efficient Fine-Tuning mit der PEFT-Bibliothek
5. **Vergleich**: Vorher/Nachher-Analyse von Modellantworten

### Nächste Schritte

- **Grösserer Datensatz**: Mehr Trainingsbeispiele für bessere Generalisierung
- **Hyperparameter-Optimierung**: Experimentiere mit verschiedenen r-Werten
- **Andere Modelle**: Teste mit anderen Basismodellen (Llama, Mistral, etc.)
- **Deployment**: LoRA-Adapter in Produktion mit Ollama oder vLLM nutzen
- **Evaluation**: BLEU, ROUGE oder menschliche Evaluation durchführen

### Ressourcen

- PEFT Dokumentation: https://huggingface.co/docs/peft
- bitsandbytes: https://github.com/TimDettmers/bitsandbytes
- LoRA Paper: https://arxiv.org/abs/2106.09685
- QLoRA Paper: https://arxiv.org/abs/2305.14314

---
**Ende des Notebooks**

Viel Erfolg beim Experimentieren!

In [ ]:
# Aufräumen - Optional
# model = None
# peft_model = None
# original_model = None
# import gc
# gc.collect()
# torch.cuda.empty_cache()

print("Notebook abgeschlossen. Die LoRA-Adapter wurden gespeichert unter: ./lora_adapters")